In [0]:
# Run after the banking pipeline completes.
# Integrity/reconciliation failures fail the notebook and Job task.

import re

# Job task parameters populate these widgets.
# Empty defaults ensure missing configuration fails clearly.
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("env", "")

CATALOG = dbutils.widgets.get("catalog").strip()
ENV = dbutils.widgets.get("env").strip().lower()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError(
        "Provide a valid catalog parameter, such as banking_workspace"
    )

if ENV not in {"dev", "prod"}:
    raise ValueError("Provide env as 'dev' or 'prod'")

SILVER = f"{CATALOG}.silver_{ENV}"
GOLD = f"{CATALOG}.gold_{ENV}"

print(f"Validating Silver: {SILVER}")
print(f"Validating Gold: {GOLD}")


def assert_no_rows(check_name, query):
    violations = spark.sql(query)

    if violations.limit(1).count() > 0:
        print(f"FAILED: {check_name}")
        display(violations.limit(20))
        raise RuntimeError(f"Validation failed: {check_name}")

    print(f"PASSED: {check_name}")


# ------------------------------------------------------------
# 1. At most one active historical version per transaction
# ------------------------------------------------------------

assert_no_rows(
    "One active version per transaction",
    f"""
    SELECT transaction_id, COUNT(*) AS active_versions
    FROM {SILVER}.transactions_history
    WHERE __END_AT IS NULL
    GROUP BY transaction_id
    HAVING COUNT(*) > 1
    """
)


# ------------------------------------------------------------
# 2. History boundaries must be valid
# ------------------------------------------------------------

assert_no_rows(
    "Valid history boundaries",
    f"""
    SELECT transaction_id, __START_AT, __END_AT
    FROM {SILVER}.transactions_history
    WHERE transaction_id IS NULL
       OR __START_AT IS NULL
       OR (
           __END_AT IS NOT NULL
           AND __END_AT <= __START_AT
       )
    """
)


# ------------------------------------------------------------
# 3. Historical versions must not overlap
#    Gaps are allowed: a transaction may be deleted and reinserted.
# ------------------------------------------------------------

assert_no_rows(
    "No overlapping historical versions",
    f"""
    WITH versions AS (
        SELECT
            transaction_id,
            __START_AT,
            __END_AT,
            LEAD(__START_AT) OVER (
                PARTITION BY transaction_id
                ORDER BY __START_AT
            ) AS next_start
        FROM {SILVER}.transactions_history
    )
    SELECT *
    FROM versions
    WHERE next_start IS NOT NULL
      AND (
          __END_AT IS NULL
          OR __END_AT > next_start
      )
    """
)


# ------------------------------------------------------------
# 4. Daily Gold counts and monetary totals must match Silver
# ------------------------------------------------------------

assert_no_rows(
    "Daily Gold reconciles with Silver",
    f"""
    WITH expected AS (
        SELECT
            business_date,
            currency,
            COUNT(*) AS transaction_count,

            SUM(
                CASE
                    WHEN status = 'SUCCESS'
                     AND direction = 'CREDIT'
                    THEN amount ELSE 0
                END
            ) AS total_credit,

            SUM(
                CASE
                    WHEN status = 'SUCCESS'
                     AND direction = 'DEBIT'
                    THEN amount ELSE 0
                END
            ) AS total_debit

        FROM {SILVER}.transactions_current
        GROUP BY business_date, currency
    )
    SELECT
        COALESCE(s.business_date, g.business_date) AS business_date,
        COALESCE(s.currency, g.currency) AS currency,
        s.transaction_count AS silver_count,
        g.transaction_count AS gold_count,
        s.total_credit AS silver_credit,
        g.total_credit AS gold_credit,
        s.total_debit AS silver_debit,
        g.total_debit AS gold_debit,
        g.net_movement AS gold_net_movement
    FROM expected s
    FULL OUTER JOIN {GOLD}.daily_transaction_summary g
        ON s.business_date <=> g.business_date
       AND s.currency <=> g.currency
    WHERE NOT (s.transaction_count <=> g.transaction_count)
       OR NOT (s.total_credit <=> g.total_credit)
       OR NOT (s.total_debit <=> g.total_debit)
       OR NOT (
           (s.total_credit - s.total_debit) <=> g.net_movement
       )
    """
)


# ------------------------------------------------------------
# 5. Report missing account relationships
#    Informational here; does not fail the task.
# ------------------------------------------------------------

missing_accounts = spark.table(
    f"{SILVER}.transactions_missing_accounts"
)

missing_count = missing_accounts.count()

print(f"Missing account relationships: {missing_count}")

if missing_count > 0:
    display(missing_accounts.limit(20))


# ------------------------------------------------------------
# 6. Report quarantine counts
#    Invalid input records are expected in our test fixtures.
# ------------------------------------------------------------

for entity in [
    "customers",
    "accounts",
    "branches",
    "merchants",
    "transactions"
]:
    rejected_count = spark.table(
        f"{SILVER}.{entity}_quarantine"
    ).count()

    print(f"{entity} quarantine records: {rejected_count}")


print("All configured integrity and reconciliation checks passed.")

In [0]:
dbutils.widgets.text("simulate_failure", "false")

simulate_failure = (
    dbutils.widgets.get("simulate_failure")
    .strip()
    .lower()
)

if simulate_failure not in {"true", "false"}:
    raise ValueError("simulate_failure must be 'true' or 'false'")

if simulate_failure == "true":
    if ENV != "dev":
        raise RuntimeError("Failure simulation is allowed only in dev")

    raise RuntimeError(
        "DELIBERATE TEST FAILURE: practising retries and Job repair"
    )